In [ ]:
# install required packages
!pip install pandas openpyxl tqdm --quiet

In [ ]:
#Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Configuration
import pandas as pd
import json
import re
from tqdm import tqdm
import os

In [ ]:
INPUT_EXCEL = "/content/drive/MyDrive/WikiHow_Project/50K_Datasets.xlsx"
OUTPUT_JSON = "/content/drive/MyDrive/WikiHow_Project/llama_finetune_dataset.json"

TEXT_COLUMN = "text"

# Minimum sentence length
MIN_LEN = 15

# Allowed labels
LABELS = {
    "Simple",
    "Mandatory",
    "Conditional",
    "Sequential",
    "Exclusive",
    "Goal-based"
}

In [ ]:
# Checking whether the file exists
if not os.path.exists(INPUT_EXCEL):
    raise FileNotFoundError(f"Excel file not found: {INPUT_EXCEL}")

os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)

In [ ]:
# Sentence splitter 

def split_sentences(text):
    if not isinstance(text, str):
        return []

    text = text.replace("\n", " ").strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)

    return [
        s.strip()
        for s in sentences
        if len(s.strip()) >= MIN_LEN
    ]

In [ ]:
#Rule-based classifier
def classify(sentence):
    s = sentence.lower()

    # Conditional
    if re.search(r"\b(if|when|unless|provided that|in case)\b", s):
        return "Conditional"

    # Mandatory
    if re.search(r"\b(must|ensure|always|never|do not|required to|should)\b", s):
        return "Mandatory"

    # Sequential
    if re.search(r"\b(first|then|after|before|next|finally|once)\b", s):
        return "Sequential"

    # Exclusive
    if re.search(r"\b(either|only one|instead of|rather than|exclusive)\b", s):
        return "Exclusive"

    # Goal-based
    if re.search(r"\b(so that|in order to|to achieve|with the goal of)\b", s):
        return "Goal-based"

    # Default
    return "Simple"

In [ ]:
# Loading Excel file
df = pd.read_excel(INPUT_EXCEL)

if TEXT_COLUMN not in df.columns:
    raise ValueError(f"Column '{TEXT_COLUMN}' not found in Excel")

texts = df[TEXT_COLUMN].dropna().astype(str).tolist()

print(f"Loaded {len(texts)} rows from Excel")

In [ ]:
#Processing and saving to JSON
dataset = []
label_count = {label: 0 for label in LABELS}

for text in tqdm(texts, desc="Processing rows"):
    sentences = split_sentences(text)

    for sent in sentences:
        label = classify(sent)

        dataset.append({
            "input": sent,
            "output": label
        })

        label_count[label] += 1

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

In [ ]:
#Final report
print("\DATASET CREATED SUCCESSFULLY")
print(f"Output file: {OUTPUT_JSON}")
print(f"Total samples: {len(dataset)}\n")

for k, v in label_count.items():
    print(f"{k:12}: {v}")

print(f"\nFile exists: {os.path.exists(OUTPUT_JSON)}")